# Fase 2 — Validação do Simulador SimPy (Real vs Simulado)
**PPGCC/UFPI — Projeto de Redes de Computadores 2026-1**

**Aluno:** Manoel Messias Pereira Medeiros  
**Matrícula:** 20251014777

Este notebook carrega os CSVs gerados pelo simulador `simulador_rudp.py`
(SimPy) e produz os gráficos comparativos Real vs Simulado exigidos
pelas 10 tarefas de validação da Fase 2.

In [ ]:
# Celula 1 - Instalacao e configuracao
!pip install plotly seaborn pandas matplotlib -q

import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pio.renderers.default = 'colab'
os.makedirs('graficos_fase2', exist_ok=True)

def salvar(fig, nome):
    path = f'graficos_fase2/{nome}'
    fig.write_html(path)
    print(f'Salvo: {path}')

print('Pronto!')

In [ ]:
# Celula 2 - Upload dos CSVs gerados pelo simulador
# Faca upload dos 10 arquivos tarefaN_*.csv gerados pelo simulador_rudp.py
from google.colab import files
uploaded = files.upload()
print('Arquivos enviados:', list(uploaded.keys()))

In [ ]:
# Celula 3 - Carrega todos os CSVs em DataFrames
t1 = pd.read_csv('tarefa1_modelagem_atraso.csv')
t2 = pd.read_csv('tarefa2_perda_bernoulli.csv')
t3 = pd.read_csv('tarefa3_timeout_retransmissoes.csv')
t4 = pd.read_csv('tarefa4_curva_vazao.csv')
t5 = pd.read_csv('tarefa5_sensibilidade_janela.csv')
t6 = pd.read_csv('tarefa6_validacao_rtt.csv')
t7 = pd.read_csv('tarefa7_impacto_jitter.csv')
t8 = pd.read_csv('tarefa8_cenario_estresse.csv')
t9 = pd.read_csv('tarefa9_analise_eficiencia.csv')
t10 = pd.read_csv('tarefa10_convergencia_ic95.csv')

print('Todos os CSVs carregados com sucesso!')
print(f'Tarefa 1: {len(t1)} linhas | Tarefa 4: {len(t4)} linhas | Tarefa 5: {len(t5)} linhas')

## Tarefa 1 — Modelagem de Atraso (Distribuição Normal)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(name='RTT Real (ms)', x=t1['cenario'], y=t1['rtt_real_ms'],
                      marker_color='#1565C0',
                      error_y=dict(type='data', array=t1['rtt_real_std_ms'])))
fig.add_trace(go.Bar(name='RTT Simulado (ms)', x=t1['cenario'], y=t1['rtt_simulado_ms'],
                      marker_color='#D84315',
                      error_y=dict(type='data', array=t1['rtt_simulado_std_ms'])))
fig.update_layout(
    title='<b>Tarefa 1 - RTT Real vs Simulado (Distribuicao Normal)</b>',
    xaxis_title='Cenario', yaxis_title='RTT (ms)',
    barmode='group', height=450, plot_bgcolor='white'
)
fig.show()
salvar(fig, 'tarefa1_rtt_real_vs_simulado.html')
print(t1.to_string(index=False))

## Tarefa 2 — Modelo de Perda de Bernoulli

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(name='Perda configurada (tc)', x=t2['cenario'], y=t2['perda_tc_pct'],
                      marker_color='#1565C0'))
fig.add_trace(go.Bar(name='Perda simulada (Bernoulli)', x=t2['cenario'], y=t2['perda_simulada_pct'],
                      marker_color='#D84315'))
fig.update_layout(
    title='<b>Tarefa 2 - Perda Configurada (tc) vs Simulada (Bernoulli)</b>',
    xaxis_title='Cenario', yaxis_title='Taxa de Perda (%)',
    barmode='group', height=450, plot_bgcolor='white'
)
fig.show()
salvar(fig, 'tarefa2_perda_bernoulli.html')
print(t2.to_string(index=False))

## Tarefa 3 — Retransmissões: Real vs Simulado

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(name='Retransmissoes Reais (tcpdump)', x=t3['cenario'],
                      y=t3['retransmissoes_reais'], marker_color='#1565C0'))
fig.add_trace(go.Bar(name='Retransmissoes Simuladas', x=t3['cenario'],
                      y=t3['retransmissoes_simuladas'], marker_color='#D84315'))
fig.update_layout(
    title='<b>Tarefa 3 - Retransmissoes: Real (Fase 1) vs Simulado (SimPy)</b>',
    xaxis_title='Cenario', yaxis_title='Numero de Retransmissoes',
    barmode='group', height=450, plot_bgcolor='white'
)
fig.show()
salvar(fig, 'tarefa3_retransmissoes.html')

fig2 = go.Figure()
fig2.add_trace(go.Bar(name='Throughput Real (Mbps)', x=t3['cenario'],
                       y=t3['throughput_real_mbps'], marker_color='#1565C0'))
fig2.add_trace(go.Bar(name='Throughput Simulado (Mbps)', x=t3['cenario'],
                       y=t3['throughput_simulado_mbps'], marker_color='#D84315'))
fig2.update_layout(
    title='<b>Tarefa 3 - Throughput: Real vs Modelo Teorico Go-Back-N</b>',
    xaxis_title='Cenario', yaxis_title='Throughput (Mbps)', yaxis_type='log',
    barmode='group', height=450, plot_bgcolor='white'
)
fig2.show()
salvar(fig2, 'tarefa3_throughput_real_vs_teorico.html')
print(t3.to_string(index=False))

## Tarefa 4 — Curva de Vazão (1MB a 100MB)

In [ ]:
fig = px.line(t4, x='tamanho_mb', y='throughput_mbps', color='cenario',
              markers=True,
              title='<b>Tarefa 4 - Curva de Vazao Simulada (1MB a 100MB)</b>',
              labels={'tamanho_mb': 'Tamanho do Arquivo (MB)',
                      'throughput_mbps': 'Throughput (Mbps)', 'cenario': 'Cenario'},
              color_discrete_map={'A': '#43A047', 'B': '#FB8C00', 'C': '#E53935'})
fig.update_layout(height=450, plot_bgcolor='white')
fig.show()
salvar(fig, 'tarefa4_curva_vazao.html')

fig2 = px.line(t4, x='tamanho_mb', y='retransmissoes', color='cenario',
               markers=True,
               title='<b>Tarefa 4 - Retransmissoes vs Tamanho do Arquivo</b>',
               labels={'tamanho_mb': 'Tamanho do Arquivo (MB)', 'cenario': 'Cenario'},
               color_discrete_map={'A': '#43A047', 'B': '#FB8C00', 'C': '#E53935'})
fig2.update_layout(height=450, plot_bgcolor='white')
fig2.show()
salvar(fig2, 'tarefa4_retransmissoes_vs_tamanho.html')

## Tarefa 5 — Sensibilidade da Janela (Saturação Teórica)

In [ ]:
fig = px.line(t5, x='window_size', y='throughput_mbps', color='cenario',
              markers=True, log_x=True,
              title='<b>Tarefa 5 - Throughput vs Tamanho da Janela (N)</b>',
              labels={'window_size': 'Tamanho da Janela N (escala log)',
                      'throughput_mbps': 'Throughput (Mbps)', 'cenario': 'Cenario'},
              color_discrete_map={'A': '#43A047', 'B': '#FB8C00', 'C': '#E53935'})

saturados = t5[t5['saturado'] == True]
fig.add_trace(go.Scatter(
    x=saturados['window_size'], y=saturados['throughput_mbps'],
    mode='markers', marker=dict(size=14, color='black', symbol='x'),
    name='Ponto de Saturacao'
))
fig.update_layout(height=500, plot_bgcolor='white')
fig.show()
salvar(fig, 'tarefa5_sensibilidade_janela.html')
print(t5.to_string(index=False))

## Tarefa 6 — Validação de RTT (Simulado vs tcpdump/ping)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(name='RTT Real (tcpdump/ping)', x=t6['cenario'],
                      y=t6['rtt_real_ms'], marker_color='#1565C0'))
fig.add_trace(go.Bar(name='RTT Simulado', x=t6['cenario'],
                      y=t6['rtt_simulado_medio_ms'], marker_color='#D84315',
                      error_y=dict(type='data', array=t6['rtt_simulado_std_ms'])))
fig.update_layout(
    title='<b>Tarefa 6 - Validacao de RTT: Simulado vs Real</b>',
    xaxis_title='Cenario', yaxis_title='RTT (ms)',
    barmode='group', height=450, plot_bgcolor='white'
)
fig.show()
salvar(fig, 'tarefa6_validacao_rtt.html')
print(t6.to_string(index=False))

## Tarefa 7 — Impacto do Jitter na Estabilidade

In [ ]:
fig = make_subplots(rows=1, cols=2,
                     subplot_titles=['RTT Medio vs Jitter', 'Coeficiente de Variacao vs Jitter'])

fig.add_trace(go.Scatter(x=t7['jitter_ms'], y=t7['rtt_medio_ms'],
                          mode='lines+markers', name='RTT Medio',
                          line=dict(color='#1565C0', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=t7['jitter_ms'], y=t7['coef_variacao_pct'],
                          mode='lines+markers', name='Coef. Variacao (%)',
                          line=dict(color='#D84315', width=3)), row=1, col=2)

fig.update_xaxes(title_text='Jitter (ms)', row=1, col=1)
fig.update_xaxes(title_text='Jitter (ms)', row=1, col=2)
fig.update_yaxes(title_text='RTT Medio (ms)', row=1, col=1)
fig.update_yaxes(title_text='Coef. Variacao (%)', row=1, col=2)
fig.update_layout(title='<b>Tarefa 7 - Impacto do Jitter na Estabilidade do Fluxo</b>',
                   height=450, plot_bgcolor='white', showlegend=False)
fig.show()
salvar(fig, 'tarefa7_impacto_jitter.html')
print(t7.to_string(index=False))

## Tarefa 8 — Cenário de Estresse (25% de Perda)

In [ ]:
print('=== Cenario de Estresse: 25% de perda, 100ms delay ===')
print(t8.to_string(index=False))
print()

comparacao = pd.DataFrame({
    'cenario': ['B (10% real)', 'C (20% real)', 'D (25% simulado)'],
    'perda_pct': [10, 20, 25],
    'tempo_s': [620.822, 1345.906, t8['tempo_simulado_s'].iloc[0]]
})

fig = px.bar(comparacao, x='cenario', y='tempo_s',
             title='<b>Tarefa 8 - Cenario de Estresse: Extrapolacao de Tempo</b>',
             labels={'tempo_s': 'Tempo de Transferencia (s)', 'cenario': 'Cenario'},
             color='cenario',
             color_discrete_sequence=['#FB8C00', '#E53935', '#6A1B9A'],
             text='tempo_s')
fig.update_traces(texttemplate='%{text:.1f}s', textposition='outside')
fig.update_layout(height=450, plot_bgcolor='white', showlegend=False)
fig.show()
salvar(fig, 'tarefa8_cenario_estresse.html')

## Tarefa 9 — Análise de Eficiência (Dados vs ACKs)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(name='Eficiencia Real (%)', x=t9['cenario'],
                      y=t9['eficiencia_real_pct'], marker_color='#1565C0'))
fig.add_trace(go.Bar(name='Eficiencia Simulada (%)', x=t9['cenario'],
                      y=t9['eficiencia_simulada_pct'], marker_color='#D84315'))
fig.update_layout(
    title='<b>Tarefa 9 - Eficiencia: Real vs Simulado</b>',
    xaxis_title='Cenario', yaxis_title='Eficiencia (%)',
    barmode='group', height=450, plot_bgcolor='white'
)
fig.show()
salvar(fig, 'tarefa9_eficiencia.html')

fig2 = px.bar(t9, x='cenario', y='razao_dados_ack',
              title='<b>Tarefa 9 - Razao Pacotes de Dados / ACKs</b>',
              labels={'razao_dados_ack': 'Razao Dados/ACK', 'cenario': 'Cenario'},
              color='cenario', color_discrete_sequence=['#43A047', '#FB8C00', '#E53935'],
              text='razao_dados_ack')
fig2.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig2.update_layout(height=450, plot_bgcolor='white', showlegend=False)
fig2.show()
salvar(fig2, 'tarefa9_razao_dados_ack.html')
print(t9.to_string(index=False))

## Tarefa 10 — Convergência Estatística (IC 95%, n=30)

In [ ]:
fig = go.Figure()
for _, row in t10.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['cenario']], y=[row['throughput_medio_mbps']],
        error_y=dict(
            type='data',
            array=[row['ic95_superior_mbps'] - row['throughput_medio_mbps']],
            arrayminus=[row['throughput_medio_mbps'] - row['ic95_inferior_mbps']],
            visible=True, thickness=3, width=10
        ),
        mode='markers', marker=dict(size=14, color='#1565C0'),
        name=f"Cenario {row['cenario']}", showlegend=False
    ))

fig.update_layout(
    title='<b>Tarefa 10 - Intervalo de Confianca 95% do Throughput (n=30 execucoes)</b>',
    xaxis_title='Cenario', yaxis_title='Throughput Medio (Mbps)',
    height=450, plot_bgcolor='white'
)
fig.show()
salvar(fig, 'tarefa10_convergencia_ic95.html')
print(t10.to_string(index=False))

## Dashboard Consolidado — Real vs Simulado

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['RTT: Real vs Simulado (Tarefa 1)',
                     'Perda: tc vs Bernoulli (Tarefa 2)',
                     'Retransmissoes: Real vs Simulado (Tarefa 3)',
                     'Eficiencia: Real vs Simulado (Tarefa 9)']
)

fig.add_trace(go.Bar(x=t1['cenario'], y=t1['rtt_real_ms'], name='Real',
                      marker_color='#1565C0', legendgroup='real', showlegend=True), row=1, col=1)
fig.add_trace(go.Bar(x=t1['cenario'], y=t1['rtt_simulado_ms'], name='Simulado',
                      marker_color='#D84315', legendgroup='sim', showlegend=True), row=1, col=1)

fig.add_trace(go.Bar(x=t2['cenario'], y=t2['perda_tc_pct'], name='Real',
                      marker_color='#1565C0', legendgroup='real', showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=t2['cenario'], y=t2['perda_simulada_pct'], name='Simulado',
                      marker_color='#D84315', legendgroup='sim', showlegend=False), row=1, col=2)

fig.add_trace(go.Bar(x=t3['cenario'], y=t3['retransmissoes_reais'], name='Real',
                      marker_color='#1565C0', legendgroup='real', showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(x=t3['cenario'], y=t3['retransmissoes_simuladas'], name='Simulado',
                      marker_color='#D84315', legendgroup='sim', showlegend=False), row=2, col=1)

fig.add_trace(go.Bar(x=t9['cenario'], y=t9['eficiencia_real_pct'], name='Real',
                      marker_color='#1565C0', legendgroup='real', showlegend=False), row=2, col=2)
fig.add_trace(go.Bar(x=t9['cenario'], y=t9['eficiencia_simulada_pct'], name='Simulado',
                      marker_color='#D84315', legendgroup='sim', showlegend=False), row=2, col=2)

fig.update_layout(
    height=700, barmode='group', plot_bgcolor='white',
    title_text='<b>Dashboard Consolidado - Validacao Real vs Simulado (Fase 2)</b>'
)
fig.show()
salvar(fig, 'dashboard_consolidado_fase2.html')
print('Dashboard gerado com sucesso!')

## Conclusões da Validação

| Tarefa | Achado Principal |
|---|---|
| 1. Atraso Normal | RTT simulado reproduz o real com erro < 0,4% |
| 2. Bernoulli | Perda simulada reproduz a taxa do tc com erro < 0,2 p.p. |
| 3. Timeout | Modelo teorico GBN subestima throughput real (372 vs 4,47 Mbps) — implementacao real e mais agressiva que o pipeline classico |
| 4. Curva de Vazao | Throughput estavel por cenario, retransmissoes crescem linearmente com o tamanho do arquivo |
| 5. Sensibilidade da Janela | Saturacao identificada em N>=64 (Cenario B) e N>=32 (Cenario C) |
| 6. Validacao RTT | Erro de 3% a 16% entre RTT simulado e medido via ping |
| 7. Jitter | Coeficiente de variacao cresce de 5% a 133% conforme jitter aumenta |
| 8. Estresse 25% | Tempo previsto de ~2.680s, consistente com a tendencia de degradacao B->C |
| 9. Eficiencia | Razao dados/ACK cresce de 1,0 (A) a 2,43 (C), eficiencia cai de 96% a 40% |
| 10. Convergencia | Intervalos de confianca de 95% estreitos confirmam estabilidade estatistica do simulador (n=30) |

**Conclusao geral:** o simulador SimPy reproduz com alta fidelidade os fenomenos estocasticos basicos (atraso e perda), mas revela uma discrepancia importante entre o modelo teorico classico de Go-Back-N (que bloqueia por RTT completo a cada janela) e a implementacao real em sockets Python, que e significativamente mais eficiente. Essa e a principal contribuicao da validacao cruzada da Fase 2.